<a href="https://colab.research.google.com/github/Rogerio-mack/IMT_CD_2026/blob/main/IMT_CD_EX_32_Classification_CV_exercicio_solucao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Atividade Avaliativa 3.2 - ECM514 Ciência de Dados**

In [62]:
#@markdown Nome **COMPLETO** e RA

Nome = 'Anna Karenina' #@param {type:"string"}
RA = '22.01164-0' #@param {type:"string"}
#@markdown

suffix_file = RA.replace('.','')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

path = 'https://github.com/Rogerio-mack/IMT_CD_2026/raw/main/data/'

#@markdown Salve o seu notebook com **IMT_CD_EX_32_\<seu_nome\>.ipynb**

#@markdown Execute esta célula para prosseguir.

# **CASE**: Adult-census

Selecione o melhor modelo de classificação de `class`, entre uma regressão logística e modelos knn (k=3-8). Valores ausentes podem ser imputados seguindo a distribuição por gênero. Para avaliar os modelos empregue uma validação cruzada do modelo com 5 partições. Verifique, além da acuracidade, outras métricas do modelo. Faça a predição de dois sujeitos fictícios com os valores de *features* mais e menos frequentes.

Empregue `seed = 42` para todas as execuções randômicas e 20% de dados de teste estratificados. **Não empregar o `pipeline` scikit-learn que será visto adiante.**

In [63]:
import pandas as pd

df = pd.read_csv("https://github.com/INRIA/scikit-learn-mooc/raw/refs/heads/main/datasets/adult-census.csv")
df.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [64]:
df.value_counts('class')

,count
class,
<=50K,37155
>50K,11687


![imagem](https://github.com/Rogerio-mack/IMT_CD_2026/blob/main/pipeline_ex.png?raw=true)

### Perguntas

1. (Preparação dos dados) Informe o formato dos conjuntos de treinamento e teste.
2. (Preparação dos dados) Quantos valores nulos foram tratados?
3. (Preparação dos dados) Informe a média dos valores dos conjuntos de treinamento e teste.
4. (Seleção dos modelos) Informe os 3 melhores modelos e sua acuracidade (CV).
5. (Modelo final) Qual a acuracidade do modelo final no conjunto de teste?
6. (Modelo final) Quais as quantidades de TP e FN do modelo final no conjunto de teste para <=50K?
7. (Predição) Qual a probabilidade das classes <=50K e >50K nos dois casos de predição?

# (Preparação dos dados) Quantos valores nulos foram tratados?

In [65]:
seed = 42

In [66]:
df.isnull().sum()

,0
age,0
workclass,0
education,0
education-num,0
marital-status,0
occupation,0
relationship,0
race,0
sex,0
capital-gain,0


In [67]:
df.replace(' ?', np.nan, inplace=True)
df.isnull().sum()

,0
age,0
workclass,2799
education,0
education-num,0
marital-status,0
occupation,2809
relationship,0
race,0
sex,0
capital-gain,0


In [68]:
np.random.seed(seed)

print('\nAntes')
display(df[ df.isna().any(axis=1) ])
print(f'Número de valores ausentes: {df.isna().sum().sum()}')
print(f'Número de linhas com valores ausentes: {df[ df.isna().any(axis=1) ].shape[0]}')

for sex in df.sex.unique():
  for c in ['workclass','occupation','native-country']:

    isna_by_sex = (df[c].isna()) & (df['sex'] == sex)
    notna_by_sex = (~df[c].isna()) & (df['sex'] == sex)

    non_null_values = df[notna_by_sex][c]

    # Use np.random.choice with replacement to sample from non_null_values
    sampled_values = np.random.choice(non_null_values, size=sum(isna_by_sex), replace=True)

    # Fill the NaN values in the 'age' column with the sampled values
    df.loc[isna_by_sex, c] = sampled_values

print('\nDepois')
display(df[ df.isna().any(axis=1) ])
print(f'Número de valores ausentes: {df.isna().sum().sum()}')
print(f'Número de linhas com valores ausentes: {df[ df.isna().any(axis=1) ].shape[0]}')


Antes


,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
4,18,NaN,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K
6,29,NaN,HS-grad,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,<=50K
13,58,NaN,HS-grad,9,Married-civ-spouse,NaN,Husband,White,Male,0,0,35,United-States,<=50K
19,40,Private,Doctorate,16,Married-civ-spouse,Prof-specialty,Husband,Asian-Pac-Islander,Male,0,0,45,NaN,>50K
22,72,NaN,7th-8th,4,Divorced,NaN,Not-in-family,White,Female,0,0,6,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48811,35,NaN,Bachelors,13,Married-civ-spouse,NaN,Wife,White,Female,0,0,55,United-States,>50K
48812,30,NaN,Bachelors,13,Never-married,NaN,Not-in-family,Asian-Pac-Islander,Female,0,0,99,United-States,<=50K
48820,71,NaN,Doctorate,16,Married-civ-spouse,NaN,Husband,White,Male,0,0,10,United-States,>50K
48822,41,NaN,HS-grad,9,Separated,NaN,Not-in-family,Black,Female,0,0,32,United-States,<=50K


Número de valores ausentes: 6465
Número de linhas com valores ausentes: 3620

Depois


,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class


Número de valores ausentes: 0
Número de linhas com valores ausentes: 0


## Resposta

6465

# (Preparação dos dados) Informe o formato dos conjuntos de treinamento e teste.

In [69]:
df[['education', 'education-num']]

,education,education-num
0,11th,7
1,HS-grad,9
2,Assoc-acdm,12
3,Some-college,10
4,Some-college,10
...,...,...
48837,Assoc-acdm,12
48838,HS-grad,9
48839,HS-grad,9
48840,HS-grad,9


'education', 'education-num' são a mesma informação. 'education-num' é ordinal, não é nominal. **Não aplicamos hot encode, tratamos como variável numérica!**



In [70]:
df[['education', 'education-num']].value_counts()

,,count
education,education-num,
HS-grad,9,15784
Some-college,10,10878
Bachelors,13,8025
Masters,14,2657
Assoc-voc,11,2061
11th,7,1812
Assoc-acdm,12,1601
10th,6,1389
7th-8th,4,955


In [71]:
df = df.drop(columns='education')
df.select_dtypes(include='object').head()


,workclass,marital-status,occupation,relationship,race,sex,native-country,class
0,Private,Never-married,Machine-op-inspct,Own-child,Black,Male,United-States,<=50K
1,Private,Married-civ-spouse,Farming-fishing,Husband,White,Male,United-States,<=50K
2,Local-gov,Married-civ-spouse,Protective-serv,Husband,White,Male,United-States,>50K
3,Private,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,United-States,>50K
4,Self-emp-inc,Never-married,Sales,Own-child,White,Female,United-States,<=50K


In [72]:
df.select_dtypes(include='number')

,age,education-num,capital-gain,capital-loss,hours-per-week
0,25,7,0,0,40
1,38,9,0,0,50
2,28,12,0,0,40
3,44,10,7688,0,40
4,18,10,0,0,30
...,...,...,...,...,...
48837,27,12,0,0,38
48838,40,9,0,0,40
48839,58,9,0,0,40
48840,22,9,0,0,20


In [73]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='class')
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.20, random_state=seed)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)





## Resposta (alternativa 1)

In [74]:
X_train.shape, X_test.shape

((39073, 12), (9769, 12))

## Resposta (alternativa 2)

## Encode

In [75]:
from sklearn.preprocessing import OneHotEncoder

# Select the columns to hot encode
categorical_cols = X_train.select_dtypes(include=['object']).columns
print(f'Encode cols: ', categorical_cols)

# Initialize the OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)

# Fit and transform X_train
encoded_data = encoder.fit_transform(X_train[categorical_cols])
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_cols))
X_train = X_train.drop(columns=categorical_cols)
X_train = pd.concat([X_train, encoded_df], axis=1)

# Transform X_test
encoded_data = encoder.transform(X_test[categorical_cols])
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_cols))
X_test = X_test.drop(columns=categorical_cols)
X_test = pd.concat([X_test, encoded_df], axis=1)

display(X_train.head())
X_train.shape, X_test.shape

Encode cols:  Index(['workclass', 'marital-status', 'occupation', 'relationship', 'race',
       'sex', 'native-country'],
      dtype='object')


,age,education-num,capital-gain,capital-loss,hours-per-week,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,workclass_ Self-emp-inc,workclass_ Self-emp-not-inc,...,native-country_ Portugal,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia
0,71,9,0,0,17,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,17,6,0,0,10,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,27,9,0,0,40,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,43,9,0,0,40,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,31,13,0,0,40,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


((39073, 81), (9769, 81))

# (Preparação dos dados) Informe a média dos valores dos conjuntos de treinamento e teste.

## Resposta (alternativa 1)

In [76]:
np.asarray(X_train).mean(), np.asarray(X_test).mean()

(np.float64(15.340994523388163), np.float64(16.517339429715314))

## Scaler

In [77]:
encoded_df.columns

Index(['workclass_ Local-gov', 'workclass_ Never-worked', 'workclass_ Private',
       'workclass_ Self-emp-inc', 'workclass_ Self-emp-not-inc',
       'workclass_ State-gov', 'workclass_ Without-pay',
       'marital-status_ Married-AF-spouse',
       'marital-status_ Married-civ-spouse',
       'marital-status_ Married-spouse-absent',
       'marital-status_ Never-married', 'marital-status_ Separated',
       'marital-status_ Widowed', 'occupation_ Armed-Forces',
       'occupation_ Craft-repair', 'occupation_ Exec-managerial',
       'occupation_ Farming-fishing', 'occupation_ Handlers-cleaners',
       'occupation_ Machine-op-inspct', 'occupation_ Other-service',
       'occupation_ Priv-house-serv', 'occupation_ Prof-specialty',
       'occupation_ Protective-serv', 'occupation_ Sales',
       'occupation_ Tech-support', 'occupation_ Transport-moving',
       'relationship_ Not-in-family', 'relationship_ Other-relative',
       'relationship_ Own-child', 'relationship_ Unmarried',

In [78]:
from sklearn.preprocessing import StandardScaler

col_numbers = X_train.drop(columns=encoded_df.columns).select_dtypes(include='number').columns
print(f'Scale cols: ', col_numbers)

scaler = StandardScaler()

# X_train
X_train[col_numbers] = scaler.fit_transform(X_train[col_numbers])

# X_test
X_test[col_numbers] = scaler.transform(X_test[col_numbers])

display(X_train.head())
X_train.shape, X_test.shape

Scale cols:  Index(['age', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')


,age,education-num,capital-gain,capital-loss,hours-per-week,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,workclass_ Self-emp-inc,workclass_ Self-emp-not-inc,...,native-country_ Portugal,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia
0,2.351033,-0.419324,-0.144218,-0.220137,-1.889257,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-1.579144,-1.584910,-0.144218,-0.220137,-2.453045,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.851333,-0.419324,-0.144218,-0.220137,-0.036809,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.313164,-0.419324,-0.144218,-0.220137,-0.036809,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-0.560209,1.134791,-0.144218,-0.220137,-0.036809,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


((39073, 81), (9769, 81))

## Resposta (alternativa 2)

In [79]:
np.asarray(X_train).mean(), np.asarray(X_test).mean()

(np.float64(0.07365700099813169), np.float64(0.07292278519258266))

# (Seleção dos modelos) Informe os 3 melhores modelos e sua acuracidade (CV).

In [84]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, cross_val_score

In [85]:
models = [ LogisticRegression(max_iter=200, random_state=seed) ]
for k in range(3, 9):
  models.append(KNeighborsClassifier(k))

models

[LogisticRegression(max_iter=200, random_state=42),
 KNeighborsClassifier(n_neighbors=3),
 KNeighborsClassifier(n_neighbors=4),
 KNeighborsClassifier(),
 KNeighborsClassifier(n_neighbors=6),
 KNeighborsClassifier(n_neighbors=7),
 KNeighborsClassifier(n_neighbors=8)]

In [93]:
cv_results = {}

for model in models:
  cv_results[model] = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy').mean()


## Resposta

In [95]:
sorted_models = sorted(cv_results.items(), key=lambda item: item[1], reverse=True);

for model, accuracy in sorted_models:
    print(f"- {model}: {accuracy:.4f}")

- LogisticRegression(max_iter=200, random_state=42): 0.8508
- KNeighborsClassifier(n_neighbors=8): 0.8423
- KNeighborsClassifier(n_neighbors=7): 0.8411
- KNeighborsClassifier(n_neighbors=6): 0.8388
- KNeighborsClassifier(): 0.8382
- KNeighborsClassifier(n_neighbors=4): 0.8358
- KNeighborsClassifier(n_neighbors=3): 0.8293


# (Modelo final) Qual a acuracidade do modelo final no conjunto de teste?

In [96]:
model = LogisticRegression(max_iter=200, random_state=seed)

model = model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## Resposta

In [97]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.91      7431
        >50K       0.74      0.59      0.66      2338

    accuracy                           0.85      9769
   macro avg       0.81      0.76      0.78      9769
weighted avg       0.85      0.85      0.85      9769



In [98]:
accuracy_score(y_test, y_pred)

0.8531067663015662

# (Modelo final) Quais as quantidades de TP e FN do modelo final no conjunto de teste para <=50K?

In [100]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
cm = pd.DataFrame(cm, index=model.classes_, columns=model.classes_)
cm

,<=50K,>50K
<=50K,6943,488
>50K,947,1391


## Resposta

In [101]:
tp_50k = cm.loc[' <=50K', ' <=50K']
fn_50k = cm.loc[' <=50K', ' >50K']

print(f"True Positives para <=50K: {tp_50k}")
print(f"False Negatives para <=50K: {fn_50k}")

True Positives para <=50K: 6943
False Negatives para <=50K: 488


# (Predição) Qual a probabilidade das classes <=50K e >50K nos dois casos de predição?

In [103]:
df['workclass'].mode(len(df['workclass']))

,workclass
0,Private


In [112]:
most_frequent_subject = {}
least_frequent_subject = {}

for col in df.columns:
    most_frequent_subject[col] = df[col].mode()[0]
    least_frequent_subject[col] = df[col].value_counts().index[-1]

df_case = pd.concat([ pd.DataFrame([most_frequent_subject]), pd.DataFrame([least_frequent_subject]) ])
df_case.reset_index(drop=True, inplace=True)
df_case

,age,workclass,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,36,Private,9,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,40,United-States,<=50K
1,86,Never-worked,1,Married-AF-spouse,Armed-Forces,Other-relative,Other,Female,2387,2201,82,Holand-Netherlands,>50K


In [113]:
df_case.drop(columns='class', inplace=True)

In [114]:
encoded_data = encoder.transform(df_case[categorical_cols])
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_cols))
df_case = df_case.drop(columns=categorical_cols)
df_case = pd.concat([df_case, encoded_df], axis=1)

display(df_case.head())
df_case.shape

,age,education-num,capital-gain,capital-loss,hours-per-week,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,workclass_ Self-emp-inc,workclass_ Self-emp-not-inc,...,native-country_ Portugal,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia
0,36,9,0,0,40,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,86,1,2387,2201,82,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


(2, 81)

In [117]:
df_case[col_numbers] = scaler.transform(df_case[col_numbers])

display(df_case.head())


,age,education-num,capital-gain,capital-loss,hours-per-week,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,workclass_ Self-emp-inc,workclass_ Self-emp-not-inc,...,native-country_ Portugal,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia
0,-0.196304,-0.419324,-0.144218,-0.220137,-0.036809,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,3.442749,-3.527554,0.181323,5.166304,3.345923,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Resposta

In [122]:
results = pd.DataFrame(model.predict_proba(df_case), columns=model.classes_)
results

,<=50K,>50K
0,0.681691,0.318309
1,0.629369,0.370631


In [124]:
model.predict(df_case)

array([' <=50K', ' <=50K'], dtype=object)